In [ ]:
class ScalarMix(nn.Module):
    def __init__(
        self, 
        mixture_size: int=13, 
        do_layer_norm: bool = True,
        initial_scalar_parameters: List[float] = None,
        have_temperature: bool = False,
        trainable: bool = True
        ) -> None:
        """
        Layer mixing through global scalar weights. Adapted from [https://github.com/allenai/allennlp/blob/main/allennlp/modules/scalar_mix.py].
        Important implemenation strategy:
            - do_layer_norm: perform layer normalization for numerical stability.
            - have_temperature: introduce an additional temp parameter that regulates the scalar weights. tau = base_temp + factor * temp
            - trainable: whether the gamma parameter is trainable.
        """
        
        super().__init__()

        self.mixture_size = mixture_size
        self.do_layer_norm = do_layer_norm

        if initial_scalar_parameters is None:
            initial_scalar_parameters = torch.zeros((mixture_size,)) # zero initialization
        
        self.softmax = nn.Softmax(0)

        if not trainable:
            self.register_buffer('gamma', torch.tensor([1.0])) # If not trainable, Gamma is fixed
        else:
            self.gamma = nn.Parameter(torch.tensor([1.0]))


        self.have_temperature = have_temperature
        if self.have_temperature:
            # adding a new learnable parameter: temperature
            self.base_temp = 1e2
            self.factor = 1e5
            self.temp = nn.Parameter(torch.tensor([1e-2]))
            
        self.scalar_weights = nn.Parameter(
            initial_scalar_parameters
            )
    
    def forward(
        self, 
        tensors: List[torch.Tensor],
        mask: torch.bool=None
        ) -> torch.Tensor:
        """
        Values:
            tensors = [layer_1, layer_2, ..., layer_n]
            weights = [w_1, ..., w_n]
        Formula:
            tau = base_temp + factor * temp
            normed_weights = layer_norm(weights * tau)
            output = gamma * sum(softmax(normed_weights) * tensors)
        Return:
            a batched tensor of the shape (batch_size, seq_len, hidden_size)
        """
        # gamma * sum(softmax(weight) * tensor, dim=1)

        if self.have_temperature:
            # tau = base_temp + factor * temp
            # weight = scalar_weight * tau
            tau = self.base_temp + self.factor*self.temp
            normed_weights = self.softmax(self.scalar_weights * tau) # (mixture_size)
        else:
            normed_weights = self.softmax(self.scalar_weights) # (mixture_size)
        
        # Elmo-style global layer norm. Adapted from AllenNLP's Scalar Mix.
        def _Elmo_do_layer_norm(tensors, broadcast_mask, num_elements_not_masked, eps = 1e-13):
            # (x - mean) / sqrt(variance + eps)
            masked_pt = broadcast_mask * tensors # (n_layer, batch_size, seq_len, hidden_size)
            num_elements_not_masked = num_elements_not_masked[None, :, None] # (1, batch_size, 1)
            mean = torch.sum(masked_pt, dim=-2) / num_elements_not_masked # (n_layer, batch_size, hidden_size)
            variance = torch.sum(((masked_pt - mean.unsqueeze(-2)) * broadcast_mask) **2, dim=-2) / num_elements_not_masked # (n_layer, batch_size, hidden_size)
            return (masked_pt - mean.unsqueeze(-2)) / torch.sqrt(variance.unsqueeze(-2) + eps) # (n_layer, batch_size, seq_len, hidden_size)

        mixed_tensor = None
        if self.do_layer_norm:
            assert mask is not None
            broadcast_mask = mask[None, :, :, None] # (1, batch_size, seq_len, 1)
            
            # Token-wise layer norm
            tensors = torch.stack(tensors, 0) # (n_layer, batch_size, seq_len, hidden_size)
            
            # PyTorch's provides off-the-shelf layer_norm function that is readily vectorized.  
            normed_tensors = F.layer_norm(tensors, normalized_shape=(tensors.size(-1), )) # tokenwise normalization: normalize across the last dimension only.
            # seq_len = tensors.size(-2) 
            # normed_tensors = F.layer_norm(tensors, normalized_shape=(tensors.size(-2), tensors.size(-1))) # sequencewise normalization: normalize across the last two dimensions
            normed_tensors = normed_tensors * broadcast_mask # masking normed tensors
            
            weights_shape = (-1,) + (1,) * (tensors.dim() - 1) # (-1, 1, 1, 1)
            normed_weights = normed_weights.view(weights_shape) # broadcastable to tensors
            mixed_tensor = torch.sum(normed_weights * normed_tensors, dim=0) # (1, batch_size, seq_len, hidden_size)

        else:
            # if no layer norm, by default we extract only the first token
            tensors = [layer[:, 0, :] for layer in tensors] # extract the first token (batch_size, seq_len, hidden_size) -> (batch_size, hidden_dim)
            tensors = torch.stack(tensors, 0) # (layer, batch_size, hidden_dim)
            tensors = F.layer_norm(tensors, normalized_shape=(tensors.size(-1),)) # normalize within token to prevent gradient explode.
            weights_shape = (-1,) + (1,) * (tensors.dim() - 1) # (-1, 1, 1)
            normed_weights = normed_weights.view(weights_shape) # broadcastable to tensors
            mixed_tensor = torch.sum(normed_weights * tensors, dim=0).unsqueeze(1) # (batch_size, 1, hidden_size)
        
        return self.gamma * mixed_tensor